# Fetch Player Data from EA Sports API
This cell connects to the EA Sports College Football API and fetches all player data for the CFB 26 national championship roster. It handles pagination to retrieve all players in batches of 100, removes duplicates, and displays the total count of players and columns.

In [110]:
import time
import requests
import pandas as pd
from math import ceil
from tqdm import tqdm

BASE = "https://drop-api.ea.com/rating/ea-sports-college-football"
HEADERS = {"User-Agent": "Mozilla/5.0"}

LOCALE = "en"
ITERATION = "cfb-26-national-championship"   # change if you want a different roster/iteration
LIMIT = 100

def fetch_page(offset: int):
    params = {
        "locale": LOCALE,
        "limit": LIMIT,
        "iteration": ITERATION,
        "offset": offset,
    }
    r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# --- first call to get totalItems ---
first = fetch_page(0)
total = first["totalItems"]
pages = ceil(total / LIMIT)

all_rows = []
seen = set()

def add_items(items):
    for it in items:
        pid = it.get("id")
        if pid is None or pid not in seen:
            all_rows.append(it)
            if pid is not None:
                seen.add(pid)

add_items(first["items"])

# --- paginate ---
for page in tqdm(range(1, pages), desc="Downloading pages"):
    offset = page * LIMIT
    payload = fetch_page(offset)
    add_items(payload["items"])
    time.sleep(0.12)  # be polite

df = pd.json_normalize(all_rows, sep="_").drop_duplicates(subset=["id"])
print("Rows:", len(df), "Cols:", df.shape[1], "Expected:", total)

Rows: 11062 Cols: 134 Expected: 11062


## Data Cleaning & Transformation
This cell cleans up the raw API data by removing unnecessary columns (like `_diff` suffixes and empty fields), simplifying stat column names, converting height to a human-readable format (e.g., 6'2"), combining first and last names, and configuring pandas display settings for better visibility.

In [111]:
df = df.drop(columns=[c for c in df.columns if c.endswith("_diff")], errors="ignore")

# Drop duplicate overall column (keep overallRating)
df = df.drop(columns=["stats_overall_value"], errors="ignore")

# Rename stats columns: stats_speed_value -> speed
df = df.rename(columns=lambda c: c.replace("stats_", "").replace("_value", ""))

# Drop placeholder columns
df = df.drop(columns=["weight", "team_isPopular"], errors="ignore")

# Add readable height string (keep height inches + height_str only)
df["height_str"] = (df["height"] // 12).astype(int).astype(str) + "'" + (df["height"] % 12).astype(int).astype(str) + '"'

# Put height and height_str next to each other
cols = df.columns.tolist()
cols.remove("height_str")
h = cols.index("height")
cols.insert(h + 1, "height_str")
df = df[cols]

# Create full name, drop first/last, move name to front
df["name"] = df["firstName"] + " " + df["lastName"]
df = df.drop(columns=["firstName", "lastName"], errors="ignore")
df = df[["name"] + [c for c in df.columns if c != "name"]]

# Display wide (no column cut-off)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

df.head()


,name,id,avatarUrl,height,height_str,overallRating,homeTown,homeState,redShirtStatus,jerseyNum,schoolYear,acceleration,agility,jumping,stamina,strength,awareness,bCVision,blockShedding,breakSack,breakTackle,carrying,catchInTraffic,catching,changeOfDirection,deepRouteRunning,finesseMoves,hitPower,impactBlocking,injury,jukeMove,kickAccuracy,kickPower,kickReturn,leadBlock,manCoverage,mediumRouteRunning,passBlock,passBlockFinesse,passBlockPower,playAction,playRecognition,powerMoves,press,pursuit,runBlock,runBlockFinesse,runBlockPower,runningStyle,shortRouteRunning,spectacularCatch,speed,spinMove,stiffArm,tackle,throwAccuracyDeep,throwAccuracyMid,throwAccuracyShort,throwOnTheRun,throwPower,throwUnderPressure,toughness,trucking,zoneCoverage,conference_id,conference_label,conference_imageUrl,team_id,team_label,team_imageUrl,position_id,position_shortLabel,position_label,position_positionType_id,position_positionType_name,iteration_id,iteration_label
0,Fernando Mendoza,4121,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/38.png?im=FaceCrop,padding=0.7",77,"6'5""",99,Miami,Florida,Previous,15,Junior,86,86,83,98,75,99,85,48,83,77,75,33,35,85,31,44,59,48,96,84,33,35,25,48,25,32,45,46,43,99,64,51,31,38,51,44,49,0,32,35,83,76,75,56,89,95,97,88,93,96,99,75,22,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png,38,Indiana,https://drop-assets.ea.com/images/4toZEtAPrEnU1X3Dqwbh93/68d698b574fe80c91a0946d56fa29da5/Indian...,QB,QB,Quarterback,offense,Offense,cfb-26-national-championship,National Championship
1,Jeremiah Smith,8726,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/72.png?im=FaceCrop,padding=0.7",75,"6'3""",98,Miami Gardens,Florida,Eligible,4,Sophomore,96,95,97,93,79,92,94,51,60,84,75,98,95,93,97,44,63,59,96,92,35,36,80,55,62,97,40,39,41,32,66,41,64,70,62,56,55,0,97,99,95,88,75,54,33,37,35,55,43,35,97,72,60,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFYCpHzWMkzHV/a3d0913ebb0be4b2a7f1002af985cd98/OhioSt...,WR,WR,Wide Receiver,offense,Offense,cfb-26-national-championship,National Championship
2,Rueben Bain Jr.,22723,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/52.png?im=FaceCrop,padding=0.7",75,"6'3""",98,Miami,Florida,Eligible,4,Junior,95,87,82,93,92,98,52,93,33,44,57,46,62,74,46,95,91,87,90,46,24,25,34,55,44,43,51,52,52,33,99,98,40,99,59,55,57,0,43,47,84,45,55,85,33,12,24,24,11,25,93,59,48,ACC,ACC,https://drop-assets.ea.com/images/63IBTgQo7jjk0Ny2g71yxn/0210a2f5caa5dd6da384ef7975aa7224/ACC.png,52,Miami,https://drop-assets.ea.com/images/4cmIyUOVj1BdzyI08TJuTV/09d58d278cc60608c23a8894a74b5c50/Miami.png,RE,REDG,Right Edge,defense,Defense,cfb-26-national-championship,National Championship
3,Caleb Downs,4674,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/72.png?im=FaceCrop,padding=0.7",72,"6'0""",97,Hoschton,Georgia,Eligible,2,Junior,95,95,91,99,74,88,77,72,57,65,68,64,75,92,47,61,94,72,97,85,32,32,87,59,87,46,49,48,47,36,92,55,88,98,53,56,55,0,47,79,93,79,75,85,37,43,45,49,48,35,96,71,85,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFYCpHzWMkzHV/a3d0913ebb0be4b2a7f1002af985cd98/OhioSt...,FS,FS,Free Safety,defense,Defense,cfb-26-national-championship,National Championship
4,David Bailey,4553,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/98.png?im=FaceCrop,padding=0.7",75,"6'3""",97,Irvine,California,Eligible,31,Senior,94,89,85,85,82,99,35,79,34,40,56,41,53,77,37,99,89,75,92,64,25,36,44,56,64,40,67,59,52,39,97,89,55,98,58,52,62,0,37,38,86,55,51,84,35,23,17,21,13,39,92,59,74,Big 12,Big 12,https://drop-assets.ea.com/images/3gwMcSmhfgpvy3vrlTmg4E/cf43c91cd3cea31f6aef33a0c3136464/Big12.png,98,Texas Tech,https://drop-a

## Rename Columns & Reorganize Data
This cell renames all columns to user-friendly names (e.g., `overallRating` → `Overall Rating`) and reorganizes them in a logical order: identity info → biographical details → team/conference/position → performance attributes → metadata. The cleaned data is saved to a CSV file.

In [112]:
# ===========================
# 1) FULL RENAME MAP
# ===========================
rename_map = {
    "name": "Name",
    "id": "Player ID",
    "avatarUrl": "Helmet URL",
    "height": "Height (in)",
    "height_str": "Height",
    "overallRating": "Overall Rating",
    "homeTown": "Hometown",
    "homeState": "Home State",
    "redShirtStatus": "Redshirt Status",
    "jerseyNum": "Jersey #",
    "schoolYear": "Class Year",

    # General
    "speed": "Speed",
    "acceleration": "Acceleration",
    "strength": "Strength",
    "agility": "Agility",
    "awareness": "Awareness",
    "jumping": "Jumping",
    "injury": "Injury",
    "stamina": "Stamina",
    "toughness": "Toughness",

    # Ballcarrier
    "carrying": "Carrying",
    "breakTackle": "Break Tackle",
    "trucking": "Trucking",
    "changeOfDirection": "Change of Direction",
    "bCVision": "Ball Carrier Vision",
    "stiffArm": "Stiff Arm",
    "spinMove": "Spin Move",
    "jukeMove": "Juke Move",
    "breakSack": "Break Sack",

    # Blocking
    "runBlock": "Run Block",
    "passBlock": "Pass Block",
    "impactBlocking": "Impact Blocking",
    "runBlockPower": "Run Block Power",
    "runBlockFinesse": "Run Block Finesse",
    "passBlockPower": "Pass Block Power",
    "passBlockFinesse": "Pass Block Finesse",
    "leadBlock": "Lead Block",

    # Passing
    "throwPower": "Throw Power",
    "throwUnderPressure": "Throw Under Pressure",
    "throwAccuracyShort": "Throw Accuracy Short",
    "throwAccuracyMid": "Throw Accuracy Mid",
    "throwAccuracyDeep": "Throw Accuracy Deep",
    "throwOnTheRun": "Throw on the Run",
    "playAction": "Play Action",

    # Defense
    "tackle": "Tackle",
    "powerMoves": "Power Moves",
    "finesseMoves": "Finesse Moves",
    "blockShedding": "Block Shedding",
    "pursuit": "Pursuit",
    "playRecognition": "Play Recognition",
    "manCoverage": "Man Coverage",
    "zoneCoverage": "Zone Coverage",
    "hitPower": "Hit Power",
    "press": "Press",

    # Receiving
    "catching": "Catching",
    "spectacularCatch": "Spectacular Catch",
    "catchInTraffic": "Catch in Traffic",
    "shortRouteRunning": "Short Route Running",
    "mediumRouteRunning": "Medium Route Running",
    "deepRouteRunning": "Deep Route Running",

    # Special Teams
    "kickAccuracy": "Kick Accuracy",
    "kickPower": "Kick Power",
    "kickReturn": "Kick Return",

    # Team / Conference / Position
    "conference_id": "Conference ID",
    "conference_label": "Conference",
    "conference_imageUrl": "Conference Logo URL",

    "team_id": "Team ID",
    "team_label": "Team",
    "team_imageUrl": "Team Logo URL",

    "position_id": "Position ID",
    "position_shortLabel": "Position (Short)",
    "position_label": "Position",
    "position_positionType_id": "Unit ID",
    "position_positionType_name": "Unit",

    # Metadata
    "iteration_id": "Iteration ID",
    "iteration_label": "Iteration",
}

df = df.rename(columns=rename_map)

# ===========================
# 2) ATTRIBUTE CATEGORY ORDER
# ===========================
general = [
    "Speed","Acceleration","Strength","Agility","Awareness",
    "Jumping","Injury","Stamina","Toughness"
]

ballcarrier = [
    "Carrying","Break Tackle","Trucking","Change of Direction","Ball Carrier Vision",
    "Stiff Arm","Spin Move","Juke Move","Break Sack"
]

blocking = [
    "Run Block","Pass Block","Impact Blocking","Run Block Power","Run Block Finesse",
    "Pass Block Power","Pass Block Finesse","Lead Block"
]

passing = [
    "Throw Power","Throw Under Pressure","Throw Accuracy Short","Throw Accuracy Mid",
    "Throw Accuracy Deep","Throw on the Run","Play Action"
]

defense = [
    "Tackle","Power Moves","Finesse Moves","Block Shedding","Pursuit","Play Recognition",
    "Man Coverage","Zone Coverage","Hit Power","Press"
]

receiving = [
    "Catching","Spectacular Catch","Catch in Traffic",
    "Short Route Running","Medium Route Running","Deep Route Running"
]

special_teams = ["Kick Accuracy","Kick Power","Kick Return"]

attr_order = (
    general + ballcarrier + blocking +
    passing + defense + receiving + special_teams
)

# ===========================
# 3) FIXED ORDER (Identity → Bio → Team → Pos → Attr → Meta)
# ===========================
identity = ["Name", "Player ID", "Helmet URL"]

bio = [
    "Height (in)", "Height", "Overall Rating",
    "Jersey #", "Class Year", "Redshirt Status",
    "Hometown", "Home State"
]

team_pos = [
    "Team ID","Team","Team Logo URL",
    "Conference ID","Conference","Conference Logo URL",
    "Position ID","Position (Short)","Position",
    "Unit","Unit ID"
]

meta = ["Iteration ID","Iteration"]

fixed_order = identity + bio + team_pos

# ===========================
# 4) FINAL COLUMN ORDER
# ===========================
fixed_set = set(fixed_order + attr_order + meta)
extras = [c for c in df.columns if c not in fixed_set]  # just in case

new_order = (
    [c for c in fixed_order if c in df.columns] +
    [c for c in attr_order if c in df.columns] +
    extras +
    [c for c in meta if c in df.columns]
)

df = df[new_order]

df.to_csv("ea_cfb26_all_players_clean.csv", index=False)

df.head()

,Name,Player ID,Helmet URL,Height (in),Height,Overall Rating,Jersey #,Class Year,Redshirt Status,Hometown,Home State,Team ID,Team,Team Logo URL,Conference ID,Conference,Conference Logo URL,Position ID,Position (Short),Position,Unit,Unit ID,Speed,Acceleration,Strength,Agility,Awareness,Jumping,Injury,Stamina,Toughness,Carrying,Break Tackle,Trucking,Change of Direction,Ball Carrier Vision,Stiff Arm,Spin Move,Juke Move,Break Sack,Run Block,Pass Block,Impact Blocking,Run Block Power,Run Block Finesse,Pass Block Power,Pass Block Finesse,Lead Block,Throw Power,Throw Under Pressure,Throw Accuracy Short,Throw Accuracy Mid,Throw Accuracy Deep,Throw on the Run,Play Action,Tackle,Power Moves,Finesse Moves,Block Shedding,Pursuit,Play Recognition,Man Coverage,Zone Coverage,Hit Power,Press,Catching,Spectacular Catch,Catch in Traffic,Short Route Running,Medium Route Running,Deep Route Running,Kick Accuracy,Kick Power,Kick Return,runningStyle,Iteration ID,Iteration
0,Fernando Mendoza,4121,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/38.png?im=FaceCrop,padding=0.7",77,"6'5""",99,15,Junior,Previous,Miami,Florida,38,Indiana,https://drop-assets.ea.com/images/4toZEtAPrEnU1X3Dqwbh93/68d698b574fe80c91a0946d56fa29da5/Indian...,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png,QB,QB,Quarterback,Offense,offense,83,86,75,86,99,83,96,98,99,75,77,75,85,85,75,76,84,83,51,45,48,49,44,43,46,48,93,96,97,95,89,88,99,56,51,44,48,38,64,25,22,59,31,35,35,33,32,32,31,33,35,25,0,cfb-26-national-championship,National Championship
1,Jeremiah Smith,8726,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/72.png?im=FaceCrop,padding=0.7",75,"6'3""",98,4,Sophomore,Eligible,Miami Gardens,Florida,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFYCpHzWMkzHV/a3d0913ebb0be4b2a7f1002af985cd98/OhioSt...,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png,WR,WR,Wide Receiver,Offense,offense,95,96,79,95,92,97,96,93,97,75,84,72,93,94,75,88,92,60,62,40,59,55,56,41,39,55,43,35,35,37,33,55,32,54,41,44,51,70,66,62,60,63,64,95,99,98,97,97,97,35,36,80,0,cfb-26-national-championship,National Championship
2,Rueben Bain Jr.,22723,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/52.png?im=FaceCrop,padding=0.7",75,"6'3""",98,4,Junior,Eligible,Miami,Florida,52,Miami,https://drop-assets.ea.com/images/4cmIyUOVj1BdzyI08TJuTV/09d58d278cc60608c23a8894a74b5c50/Miami.png,ACC,ACC,https://drop-assets.ea.com/images/63IBTgQo7jjk0Ny2g71yxn/0210a2f5caa5dd6da384ef7975aa7224/ACC.png,RE,REDG,Right Edge,Defense,defense,84,95,92,87,98,82,90,93,93,57,44,59,74,52,55,45,46,33,59,51,87,57,55,52,52,55,11,25,24,12,33,24,33,85,98,95,93,99,99,44,48,91,40,62,47,46,43,43,46,24,25,34,0,cfb-26-national-championship,National Championship
3,Caleb Downs,4674,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/72.png?im=FaceCrop,padding=0.7",72,"6'0""",97,2,Junior,Eligible,Hoschton,Georgia,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFYCpHzWMkzHV/a3d0913ebb0be4b2a7f1002af985cd98/OhioSt...,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png,FS,FS,Free Safety,Defense,defense,93,95,74,95,88,91,97,99,96,68,65,71,92,77,75,79,85,57,53,49,72,55,56,47,48,59,48,35,45,43,37,49,36,85,55,61,72,98,92,87,85,94,88,75,79,64,47,46,47,32,32,87,0,cfb-26-national-championship,National Championship
4,David Bailey,4553,"https://ratings-images-prod.pulse.ea.com/college-football-26/helmets/98.png?im=FaceCrop,padding=0.7",75,"6'3""",97,31,Senior,Eligible,Irvine,California,98,Texas Tech,https://drop-assets.ea.com/images/2Lxte8K570VxzhDqT7gPqO/3b7e01ad825eab235b7b517a069d5490/TexasT...,Big 12,Big 12,https://drop-assets.ea.com/images/3gwMcSmhfgpvy3vrlTmg4E/cf43c91cd3cea31f6aef33a0c3136464/Big12.png,LE,LEDG,Left Edge,Defense,defense,86,94,82,89,99,85,92,85,92,56,4

## Load and Display Initial Player Data

This cell loads the cleaned EA Sports player data from CSV and converts image URLs into HTML image tags for visual display. It serves as a preview of the raw data structure before adding headshots.


In [113]:
from IPython.display import HTML

df = pd.read_csv("ea_cfb26_all_players_clean.csv")

def to_img(url, size=60):
    if pd.isna(url) or url == "":
        return ""
    return f'<img src="{url}" width="{size}">'

image_cols = [
    "Helmet URL",
    "Team Logo URL",
    "https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png",
    "Conference Logo URL"
]

for col in image_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: to_img(x, size=60))

# Show only first 5 rows with images
HTML(df.head().to_html(escape=False))


,Name,Player ID,Helmet URL,Height (in),Height,Overall Rating,Jersey #,Class Year,Redshirt Status,Hometown,Home State,Team ID,Team,Team Logo URL,Conference ID,Conference,Conference Logo URL,Position ID,Position (Short),Position,Unit,Unit ID,Speed,Acceleration,Strength,Agility,Awareness,Jumping,Injury,Stamina,Toughness,Carrying,Break Tackle,Trucking,Change of Direction,Ball Carrier Vision,Stiff Arm,Spin Move,Juke Move,Break Sack,Run Block,Pass Block,Impact Blocking,Run Block Power,Run Block Finesse,Pass Block Power,Pass Block Finesse,Lead Block,Throw Power,Throw Under Pressure,Throw Accuracy Short,Throw Accuracy Mid,Throw Accuracy Deep,Throw on the Run,Play Action,Tackle,Power Moves,Finesse Moves,Block Shedding,Pursuit,Play Recognition,Man Coverage,Zone Coverage,Hit Power,Press,Catching,Spectacular Catch,Catch in Traffic,Short Route Running,Medium Route Running,Deep Route Running,Kick Accuracy,Kick Power,Kick Return,runningStyle,Iteration ID,Iteration
0,Fernando Mendoza,4121,,77,"6'5""",99,15,Junior,Previous,Miami,Florida,38,Indiana,,Big 10,Big Ten,,QB,QB,Quarterback,Offense,offense,83,86,75,86,99,83,96,98,99,75,77,75,85,85,75,76,84,83,51,45,48,49,44,43,46,48,93,96,97,95,89,88,99,56,51,44,48,38,64,25,22,59,31,35,35,33,32,32,31,33,35,25,0,cfb-26-national-championship,National Championship
1,Jeremiah Smith,8726,,75,"6'3""",98,4,Sophomore,Eligible,Miami Gardens,Florida,72,Ohio State,,Big 10,Big Ten,,WR,WR,Wide Receiver,Offense,offense,95,96,79,95,92,97,96,93,97,75,84,72,93,94,75,88,92,60,62,40,59,55,56,41,39,55,43,35,35,37,33,55,32,54,41,44,51,70,66,62,60,63,64,95,99,98,97,97,97,35,36,80,0,cfb-26-national-championship,National Championship
2,Rueben Bain Jr.,22723,,75,"6'3""",98,4,Junior,Eligible,Miami,Florida,52,Miami,,ACC,ACC,,RE,REDG,Right Edge,Defense,defense,84,95,92,87,98,82,90,93,93,57,44,59,74,52,55,45,46,33,59,51,87,57,55,52,52,55,11,25,24,12,33,24,33,85,98,95,93,99,99,44,48,91,40,62,47,46,43,43,46,24,25,34,0,cfb-26-national-championship,National Championship
3,Caleb Downs,4674,,72,"6'0""",97,2,Junior,Eligible,Hoschton,Georgia,72,Ohio State,,Big 10,Big Ten,,FS,FS,Free Safety,Defense,defense,93,95,74,95,88,91,97,99,96,68,65,71,92,77,75,79,85,57,53,49,72,55,56,47,48,59,48,35,45,43,37,49,36,85,55,61,72,98,92,87,85,94,88,75,79,64,47,46,47,32,32,87,0,cfb-26-national-championship,National Championship
4,David Bailey,4553,,75,"6'3""",97,31,Senior,Eligible,Irvine,California,98,Texas Tech,,Big 12,Big 12,,LE,LEDG,Left Edge,Defense,defense,86,94,82,89,99,85,92,85,92,56,40,59,77,35,51,55,64,34,58,67,75,62,52,52,59,56,13,39,17,23,35,21,39,84,89,99,79,98,97,64,74,89,55,53,38,41,37,40,37,25,36,44,0,cfb-26-national-championship,National Championship


## Merge Player Data with Headshots from Roster

This cell performs a comprehensive, multi-stage join to add headshot URLs from the roster data to each EA Sports player. The process handles name mismatches, team variations, and players who have transferred between schools.

**Matching Strategy (in order of priority):**
1. **Exact Match**: Normalized name + team (handles standard cases where naming is consistent)
2. **Fuzzy Team Match**: If team is unmatched, find similar team names and retry player match
3. **Cross-Team Transfer Match**: Player matched by normalized name across any team (catches players like AJ Stephens who moved from Rice to Southern)
4. **Fuzzy Cross-Team Match**: Final fallback using fuzzy string matching on normalized names across all teams

**Data Normalization:**
- Names: lowercase, remove punctuation/special chars, trim whitespace
- Teams: same plus manual mapping for known variants (e.g., Cal ↔ California, UConn ↔ Connecticut)
- Name aliases: Manual overrides for known cases (e.g., "mike sandjo" → "mike yoan sandjo-njiki")

**Cleanup:**
- Helper columns (normalized names, match flags) are removed after joining
- Final dataframe retains only original player attributes + new `Headshot URL` column
- **Output**: Saved to `ea_cfb26_players_with_headshots.csv`

In [114]:
import re
from difflib import get_close_matches

# Start from a clean base to avoid previous merged row inflation
# (if df already contains previous join results, this resets to original dataset)
df = pd.read_csv('ea_cfb26_all_players_clean.csv')

# Normalize function with common team/name normalization rules
def normalize_name(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace("(ohio)", "(oh)")
    s = re.sub(r"[^a-z0-9 ()-]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

# Manual team mappings for known mismatch cases
team_map = {
    'appalachian state': 'app state',
    'cal': 'california',
    'connecticut': 'uconn',
    'middle tennessee state': 'middle tennessee',
    'umass': 'massachusetts',
    'fiu': 'florida international',
    'fau': 'florida atlantic',
    'usf': 'south florida',
}

# Name alias mapping for normalized names that should resolve to a single roster headshot record
name_alias_map = {
    'mike sandjo': 'mike yoan sandjo-njiki',
    # add more manual alias names here if needed
}

manual_team_mapteam_map = {
    "App State": "Appalachian State",
    "California": "Cal",
    "UConn": "Connecticut",
    "Florida Atlantic": "FAU",
    "Florida International": "FIU",
    "Hawai'i": "Hawaii",
    "Miami (OH)": "Miami (Ohio)",
    "Middle Tennessee": "Middle Tennessee State",
    "San José State": "San Jose State",
    "Massachusetts": "UMass",
    "South Florida": "USF"
}

# Load headshots and normalize
headshots = pd.read_csv("rosters_clean.csv")
headshots['player_norm'] = headshots['player_name'].apply(normalize_name)
headshots['team_norm'] = headshots['team'].apply(normalize_name)
headshots['team_norm'] = headshots['team_norm'].replace(manual_team_map)
headshots['player_norm'] = headshots['player_norm'].replace(name_alias_map)

# Add normalization to main df
if 'Name' in df.columns:
    df['name_norm'] = df['Name'].apply(normalize_name)
else:
    df['name_norm'] = df['name'].apply(normalize_name)
df['name_norm'] = df['name_norm'].replace(name_alias_map)

if 'Team' in df.columns:
    df['team_norm'] = df['Team'].apply(normalize_name)
else:
    df['team_norm'] = df['team_label'].apply(normalize_name)

df['team_norm'] = df['team_norm'].replace(manual_team_map)

# Deduplicate headshots key pairs first, to avoid row explosion
headshots = headshots.drop_duplicates(subset=['player_norm', 'team_norm'])

# exact join first
df = df.merge(
    headshots[['player_norm', 'team_norm', 'headshot_url']],
    left_on=['name_norm', 'team_norm'],
    right_on=['player_norm', 'team_norm'],
    how='left'
)

# Keep headshot column and clean up
df = df.rename(columns={'headshot_url': 'Headshot URL'})

# sanity: row count should remain same after merge
print('post-merge row count:', len(df))

# Handle any duplicate columns globally
if df.columns.duplicated().any():
    duplicates = df.columns[df.columns.duplicated()].unique()
    for col in duplicates:
        candidates = df.loc[:, df.columns == col]
        df[col] = candidates.bfill(axis=1).iloc[:, 0]
    df = df.loc[:, ~df.columns.duplicated()]

# Remove possible duplicate Headshot URL columns after merge
hs_cols = [c for c in df.columns if c.lower() == 'headshot url']
if len(hs_cols) > 1:
    df['Headshot URL'] = df[hs_cols].bfill(axis=1).iloc[:, 0]
    cols_to_drop = [c for c in hs_cols if c != 'Headshot URL']
    df = df.drop(columns=cols_to_drop)

# Fuzzy team fallback for unmatched rows
unmatched = df[df['Headshot URL'].isna()].copy()
for idx, row in unmatched.iterrows():
    team = row['team_norm']
    candidate_teams = sorted(headshots['team_norm'].unique())
    team_match = get_close_matches(team, candidate_teams, n=1, cutoff=0.8)
    if team_match:
        df.at[idx, 'team_norm'] = team_match[0]

# No re-merge by full outer again; we already joined once on cleaned normalized keys
# If further unmatched remain, do fuzzy player matching only (no increasing row count)

# Final fuzzy player matching for remaining unmatched rows
unmatched = df[df['Headshot URL'].isna()].copy()
for idx, row in unmatched.iterrows():
    candidate_df = headshots[headshots['team_norm'] == row['team_norm']] if pd.notna(row['team_norm']) else headshots
    if candidate_df.empty:
        candidate_df = headshots

    # Use normalized names for reliable matching
    candidate_norms = candidate_df['player_norm'].unique().tolist()
    name_norm = row['name_norm'] if 'name_norm' in row else normalize_name(row['Name'] if 'Name' in row else row.get('name', ''))

    # 1) exact normalized name match within team scope
    exact_team = candidate_df[candidate_df['player_norm'] == name_norm]
    if not exact_team.empty:
        url = exact_team['headshot_url'].iloc[0]
        df.at[idx, 'Headshot URL'] = url
        df.at[idx, 'fuzzy_match'] = True
        df.at[idx, 'fuzzy_player_match'] = exact_team['player_name'].iloc[0]
        continue

    # 2) fuzzy normalized name match within team scope
    team_hits = get_close_matches(name_norm, candidate_norms, n=1, cutoff=0.8)
    if team_hits:
        matched_norm = team_hits[0]
        matched_row = candidate_df[candidate_df['player_norm'] == matched_norm].iloc[0]
        df.at[idx, 'Headshot URL'] = matched_row['headshot_url']
        df.at[idx, 'fuzzy_match'] = True
        df.at[idx, 'fuzzy_player_match'] = matched_row['player_name']
        continue

    # 3) fallback: any-team exact normalized name match
    any_exact = headshots[headshots['player_norm'] == name_norm]
    if not any_exact.empty:
        url = any_exact['headshot_url'].iloc[0]
        df.at[idx, 'Headshot URL'] = url
        df.at[idx, 'fuzzy_match'] = True
        df.at[idx, 'fuzzy_player_match'] = any_exact['player_name'].iloc[0]
        df.at[idx, 'team_norm'] = any_exact['team_norm'].iloc[0]
        continue

    # 4) fallback: any-team fuzzy normalized name match
    any_hits = get_close_matches(name_norm, headshots['player_norm'].unique().tolist(), n=1, cutoff=0.75)
    if any_hits:
        matched_norm = any_hits[0]
        matched_row = headshots[headshots['player_norm'] == matched_norm].iloc[0]
        df.at[idx, 'Headshot URL'] = matched_row['headshot_url']
        df.at[idx, 'fuzzy_match'] = True
        df.at[idx, 'fuzzy_player_match'] = matched_row['player_name']
        df.at[idx, 'team_norm'] = matched_row['team_norm']
        continue

# Remove unneeded helper columns produced by merging/fuzzy matching
cleanup_cols = [
    'player_norm', 'team_norm', 'name_norm',
    'fuzzy_match', 'fuzzy_player_match'
]
cleanup_cols = [c for c in cleanup_cols if c in df.columns]
if cleanup_cols:
    df = df.drop(columns=cleanup_cols)

# Summary stats
matched_new = df['Headshot URL'].notna().sum()
print(f"Total rows: {len(df)}")
print(f"Matched with headshot: {matched_new}")
print(f"Missing headshot after fuzzy matching: {(df['Headshot URL'].isna()).sum()}")

# Show some unmatched name/team combos for analysis
name_col = 'Name' if 'Name' in df.columns else 'name'
team_col = 'Team' if 'Team' in df.columns else 'team_label'

df_unmatched = df[df['Headshot URL'].isna()][[name_col, team_col]].drop_duplicates().head(100)
print(df_unmatched)


post-merge row count: 11062
Total rows: 11062
Matched with headshot: 11031
Missing headshot after fuzzy matching: 31
                      Name              Team
2164        Gabe Ervin Jr.      Kansas State
2960        Deondrae Riden         Texas A&M
3231           Noah Biglow    Louisiana Tech
3868        Tiaquelin Mims       Texas State
4274   Joseph Jefferson II        Louisville
4363         Masai Reddick         Tennessee
4621          Antwann Hill           Memphis
5101       Pokaiaua Haunga               BYU
5140         Sergio Muasau            Hawaii
5592       Kahlil Brantley               FAU
5716           Omari Okeke        Utah State
6053         Donald Chaney         Charlotte
6896          Kenzo Viteri              Navy
7277            Clay Wedin            Auburn
7633            N-Kye Wynn           Rutgers
7787            Woody Jean               FAU
7932        Clinton Mahoni               FIU
8180          Lucas Borrow            Hawaii
8393     Bennett Boehnlein  

In [115]:
# Convert headshot URLs to images
if 'Headshot URL' in df.columns:
    df['Headshot URL'] = df['Headshot URL'].apply(lambda x: to_img(x, size=60))

# Update image columns to include headshots
image_cols = [
    "Helmet URL",
    "Headshot URL",
    "Team Logo URL",
    "Conference Logo URL"
]

# Save the final merged dataframe with headshots to CSV
# (before converting URLs to HTML for display)
df.to_csv('ea_cfb26_players_with_headshots.csv', index=False)
print("✓ Final dataset saved to: ea_cfb26_players_with_headshots.csv")
print(f"Total rows: {len(df)}, Columns: {len(df.columns)}")

# Show first 5 rows with all images
HTML(df.head().to_html(escape=False))
df.info()

✓ Final dataset saved to: ea_cfb26_players_with_headshots.csv
Total rows: 11062, Columns: 78
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11062 entries, 0 to 11061
Data columns (total 78 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Name                  11062 non-null  object
 1   Player ID             11062 non-null  int64 
 2   Helmet URL            11062 non-null  object
 3   Height (in)           11062 non-null  int64 
 4   Height                11062 non-null  object
 5   Overall Rating        11062 non-null  int64 
 6   Jersey #              11062 non-null  int64 
 7   Class Year            11062 non-null  object
 8   Redshirt Status       11062 non-null  object
 9   Hometown              11062 non-null  object
 10  Home State            11062 non-null  object
 11  Team ID               11062 non-null  int64 
 12  Team                  11062 non-null  object
 13  Team Logo URL         11062 non-null  objec

## Load NFL Combine Data
This cell imports the nflreadpy library and loads NFL combine measurement and performance data from the 2026 season to enable comparison between college player ratings and professional combine metrics.

In [120]:
import pandas as pd

url = "https://github.com/nflverse/nflverse-data/releases/download/combine/combine.parquet"
combine_data = pd.read_parquet(url)
combine_2026 = combine_data[combine_data["season"] == 2026]
combine_2026 = combine_2026[['player_name', 'school', 'forty',
                                          'bench', 'vertical', 'broad_jump', 'cone', 'shuttle']]
combine_2026

,player_name,school,forty,bench,vertical,broad_jump,cone,shuttle
8649,Chris Adams,Memphis,NaN,NaN,NaN,NaN,NaN,NaN
8650,Jose Aguilar,Tennessee,NaN,NaN,NaN,NaN,NaN,NaN
8651,Drew Allar,Penn St.,NaN,NaN,NaN,NaN,NaN,NaN
8652,CJ Allen,Georgia,NaN,NaN,NaN,NaN,NaN,NaN
8653,Kaytron Allen,Penn St.,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
8963,Jeremiah Wright,Auburn,NaN,NaN,NaN,NaN,NaN,NaN
8964,Taurean York,Texas A&M,NaN,25.0,NaN,NaN,7.32,4.48
8965,Colbie Young,Georgia,4.49,NaN,NaN,NaN,NaN,NaN
8966,Zion Young,Missouri,NaN,NaN,NaN,NaN,NaN,NaN
